# Human LVAD cardiomyocytes — selection of the nuclei used for Fig. 6b


In [48]:
# Core scverse libraries
import scanpy as sc
import anndata as ad
import pandas as pd
import matplotlib.pyplot as plt
import scanpy.external as sce
import json

In [49]:
sc.settings.verbosity = 3  # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()
sc.settings.set_figure_params(
    dpi=300,
    facecolor="white",
    figsize=(10, 8),  # Adjust as needed
)
plt.rcParams['axes.grid'] = False

In [50]:
adata = sc.read_h5ad("./adata/CM_LVRR_DCM_120626.h5ad")

In [51]:
adata.obs.loc[adata.obs["sample"].str.contains("C"), "type"] = "Ctrl"
adata.obs.loc[adata.obs["sample"].str.contains("V"), "type"] = "VAD"
adata.obs.loc[adata.obs["sample"].str.contains("T"), "type"] = "HT"

## 1. Normalisation, highly variable genes, PCA

In [55]:
sc.pp.normalize_total(adata, target_sum=1e4)

normalizing counts per cell
    finished (0:00:00)


In [56]:
sc.pp.log1p(adata)

In [57]:
adata.raw = adata.copy()

In [59]:
sc.pp.highly_variable_genes(adata, n_top_genes=5000)

extracting highly variable genes
    finished (0:00:00)
--> added
    'highly_variable', boolean vector (adata.var)
    'means', float vector (adata.var)
    'dispersions', float vector (adata.var)
    'dispersions_norm', float vector (adata.var)


In [60]:
sc.pp.scale(adata, max_value=10)

/Users/takahiro/miniforge3/envs/scanpy/lib/python3.10/functools.py:889: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)


In [61]:
sc.tl.pca(adata, svd_solver="arpack")

computing PCA
    with n_comps=50


/var/folders/dh/5v73cjps0k72p3ltz6hhg22c0000gn/T/ipykernel_94229/1703553697.py:1: UserWarning: When using a mask parameter with anndata<0.9 on a dense array, the PCAcan have slightly different results due the array being column major instead of row major.
  sc.tl.pca(adata, svd_solver="arpack")


    finished (0:00:32)


In [63]:
adata.write_h5ad("./adata/CM_Allsample_analysed.h5ad")

In [253]:
adata = sc.read_h5ad("./adata/CM_Allsample_analysed.h5ad")

## 2. Harmony integration, neighbours, UMAP

In [254]:
sce.pp.harmony_integrate(adata, 'sample')

2026-06-15 12:40:38,994 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...
2026-06-15 12:40:43,299 - harmonypy - INFO - sklearn.KMeans initialization complete.
2026-06-15 12:40:43,349 - harmonypy - INFO - Iteration 1 of 10
2026-06-15 12:40:46,292 - harmonypy - INFO - Iteration 2 of 10
2026-06-15 12:40:50,263 - harmonypy - INFO - Converged after 2 iterations


In [255]:
adata

AnnData object with n_obs × n_vars = 11736 × 33158
    obs: 'sample', 'LVRR', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'doublet_score', 'predicted_doublet', 'leiden', 'cell_type', 'type'
    var: 'n_cells', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'mean', 'std'
    uns: 'cell_type_colors', 'hvg', 'leiden', 'log1p', 'neighbors', 'pca', 'sample_colors', 'scrublet', 'umap'
    obsm: 'X_pca', 'X_pca_harmony', 'X_umap'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'

In [256]:
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40, use_rep='X_pca_harmony')

computing neighbors
    finished: added to `.uns['neighbors']`
    `.obsp['distances']`, distances for each pair of neighbors
    `.obsp['connectivities']`, weighted adjacency matrix (0:00:01)


In [257]:
sc.tl.umap(adata)

computing UMAP
    finished: added
    'X_umap', UMAP coordinates (adata.obsm)
    'umap', UMAP parameters (adata.uns) (0:00:05)


## 3. Leiden clustering (resolution 1) and per-cluster inspection

In [259]:
sc.tl.leiden(adata, resolution=1)

running Leiden clustering
    finished: found 11 clusters and added
    'leiden', the cluster labels (adata.obs, categorical) (0:00:00)


In [260]:
sc.pl.umap(adata, color=["leiden","LVRR"])

In [262]:
sc.pl.dotplot(adata, [
  "TTN", "MYBPC3", # CMs
  "DCN", "COL1A1", "POSTN", # FBs
  "FRMD3", "DLC1", "MYH11", # SMCs
  "PECAM1", "CDH5", "VWF", "NPR3", # Endothelial, cardiac cells
  "PTPRC", "MRC1", "CD163", # Macrophages
  "CD3E", "SKAP1", "CD79A", "CD79B", "IL7R", "KIT", # T cell, B cell
  "LMNB1", "SLPI", "S100A9", # Granulocytes
  "NRXN1", "NRXN3", "UPK3B", "GPC3", 
  "PLIN1", "XKR4", "ACTA2", "MS4A1", "NCR1", "VTN", "COLEC11", "STEAP4", "KCNJ8", "MMRN1", "FLT4"
], groupby="leiden", vmax=5)

## 4. Exclusion of clusters 1, 5, 9, 10 → 8,447 nuclei

In [263]:
adata = adata[~adata.obs["leiden"].isin(["1","5","9","10"])]

In [264]:
adata

View of AnnData object with n_obs × n_vars = 8447 × 33158
    obs: 'sample', 'LVRR', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'doublet_score', 'predicted_doublet', 'leiden', 'cell_type', 'type'
    var: 'n_cells', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'mean', 'std'
    uns: 'cell_type_colors', 'hvg', 'leiden', 'log1p', 'neighbors', 'pca', 'sample_colors', 'scrublet', 'umap', 'type_colors', 'leiden_colors', 'LVRR_colors'
    obsm: 'X_pca', 'X_pca_harmony', 'X_umap'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'

## 5. Save (barcodes of this object define the Fig. 6b nuclei)

In [301]:
adata.write_h5ad("./adata/CM_analysed_130626.h5ad")